# Compartment taxonomy

A `Property` is a group of mutually exclusive traits. A `PropertyMap` is the
table of compartments those properties create. Queries are selectors
(`age["0-4"]`, `state["I"] & ~sev["severe"]`) that compile to integer index arrays.

Ragged maps are allowed: a property may be absent on some compartments. Absent
is Kleene *unknown*, so both `severity["mild"]` and `~severity["mild"]` skip
unstratified rows. Use `.absent()` / `.present()` to reach them.

In [ ]:
from summer4 import Property, PropertyMap

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
sev = Property("severity", ("mild", "severe"))

pm = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(sev, where=state["I"])
)

assert pm.size == 12  # S×3 + I×3×2 + R×3
assert pm.n_properties == 3
pm

In [ ]:
infected_young = pm.select(state["I"] & age["0-4"])
assert infected_young.size == 2  # mild and severe

not_severe = pm.select(age[("0-4", "5-9")] & ~sev["severe"])
assert set(not_severe.tolist()).isdisjoint(pm.select(sev["severe"]).tolist())

unstratified = pm.select(sev.absent())
assert set(unstratified.tolist()) == set(pm.select(state["S"] | state["R"]).tolist())

In [ ]:
by_age = pm.partition(age)
assert set(t.name for t in by_age) == {"0-4", "5-9", "10+"}
assert sum(idx.size for idx in by_age.values()) == pm.size

mild = set(pm.select(sev["mild"]).tolist())
not_mild = set(pm.select(~sev["mild"]).tolist())
absent = set(pm.select(sev.absent()).tolist())
assert mild | not_mild | absent == set(range(pm.size))
assert mild.isdisjoint(not_mild) and mild.isdisjoint(absent)